In [1]:
from models.gpkg import GeoPackage

In [2]:
gpkg = GeoPackage()

In [3]:
from pprint import pprint
sigmine_gdf = gpkg.read_layer('sigmine_rs')

c:\Users\adminitsd\Documents\ufrgs\mestrado\python_god\venv\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


In [3]:
from models.mapbiomas import MapBiomas

mapbio = MapBiomas()
mp_2022 = mapbio.lulc_by_year(2022)

In [4]:
sist_eco_rs = gpkg.read_layer('se_rs_clip')


In [7]:
from pprint import pprint

pprint(mp_2022.profile)

{'blockxsize': 256,
 'blockysize': 256,
 'compress': 'lzw',
 'count': 1,
 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'),
 'driver': 'GTiff',
 'dtype': 'uint8',
 'height': 146483,
 'interleave': 'band',
 'nodata': None,
 'tiled': True,
 'transform': Affine(0.0002694945852358564, 0.0, -74.02099974839176,
       0.0, -0.0002694945852358564, 5.435705784207225),
 'width': 154470}


In [ ]:
import rioxarray as rxr

mp_rx = rxr.open_rasterio(mp_2022, chunks=True)
mp_rx.size

In [10]:
gpkg = GeoPackage()

In [11]:
gdf = gpkg.read_layer('rs_aoi_polyconic')

In [ ]:
geoms = [feature["geometry"] for feature in gdf.__geo_interface__["features"]]

In [22]:
gdf = gdf.to_crs(4326)

In [23]:
geoms = [feature["geometry"] for feature in gdf.__geo_interface__["features"]]

In [26]:
from rasterio.mask import mask

clip, transform = mask(mp_2022, geoms, crop=True, all_touched=True, filled=False)

In [28]:
out_meta = mp_2022.meta.copy()
out_meta.update({
    "height": clip.shape[1],
    "width": clip.shape[2],
    "transform": transform
})

In [29]:
import rasterio
with rasterio.open('mapiomas_2022_rs.tif', 'w', **out_meta) as dest:
    dest.write(clip)